# ??? crop

- ??: ???? ?? ??? crop ???? ??? ??? ??? ??.
- ????(??): ???
- ?? ?? ??: 06
- ?? ??: https://app.notion.com/p/397bf34cc2ef80019edfd0dfc668d989

> ??? ?? ? ????? ??? ????? ??????. ??? ? ??? ??? ??? ? ?? ?????.


In [ ]:
!pip install roboflow

import os
from roboflow import Roboflow
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY ????? ?????.")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("s-workspace-o2evy").project("dyd-safe-2")
version = project.version(2)
dataset = version.download("coco")

In [ ]:
import json
from pathlib import Path
from copy import deepcopy

import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

from shapely.geometry import Polygon, box, MultiPolygon, GeometryCollection
from pycocotools import mask as mask_utils


# =========================
# 1) 경로 설정
# =========================

INPUT_ROOT = Path("/content/dyd-safe-2-2")
OUTPUT_ROOT = Path("/content/dyd-safe-2-2-hook-crops-connected")


# =========================
# 2) 주요 설정
# =========================

# Roboflow 클래스 이름과 정확히 같게 수정
HOOK_CLASS = "hook"
LIFELINE_CLASS = "lifeline"

# True: crop 안에 들어온 모든 객체 mask 저장
# False: 기준 후크 mask만 저장
KEEP_ALL_CLASSES_IN_CROP = True

# 후크 bbox 기준 여백
MARGIN_RATIO = 0.8

# 너무 작은 mask 제거 기준
MIN_MASK_AREA = 5

# "preserve": 원래 polygon이면 polygon, RLE면 RLE로 저장
# "polygon": 전부 polygon으로 저장
# "rle": 전부 RLE로 저장
OUTPUT_MASK_FORMAT = "preserve"


# =========================
# 3) 이미지 경로 찾기
# =========================

def resolve_image_path(input_dir, file_name):
    candidates = [
        input_dir / file_name,
        input_dir / "images" / file_name,
        input_dir / Path(file_name).name,
        input_dir / "images" / Path(file_name).name,
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


# =========================
# 4) crop box 계산
# =========================

def compute_crop_box_from_hook(hook_ann, image_width, image_height):
    x, y, w, h = hook_ann["bbox"]

    x1 = x
    y1 = y
    x2 = x + w
    y2 = y + h

    margin_x = w * MARGIN_RATIO
    margin_y = h * MARGIN_RATIO

    crop_x1 = max(0, int(x1 - margin_x))
    crop_y1 = max(0, int(y1 - margin_y))
    crop_x2 = min(image_width, int(x2 + margin_x))
    crop_y2 = min(image_height, int(y2 + margin_y))

    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        return None

    return crop_x1, crop_y1, crop_x2, crop_y2


# =========================
# 5) polygon 처리 함수
# =========================

def flatten_polygon_coords(coords):
    result = []
    for x, y in coords:
        result.extend([float(x), float(y)])
    return result


def polygon_area(poly_flat):
    if len(poly_flat) < 6:
        return 0.0

    coords = [(poly_flat[i], poly_flat[i + 1]) for i in range(0, len(poly_flat), 2)]
    poly = Polygon(coords)

    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        return 0.0

    return float(poly.area)


def clip_single_polygon_to_crop(poly_flat, crop_box):
    crop_x1, crop_y1, crop_x2, crop_y2 = crop_box

    if len(poly_flat) < 6:
        return []

    coords = [(poly_flat[i], poly_flat[i + 1]) for i in range(0, len(poly_flat), 2)]
    poly = Polygon(coords)

    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        return []

    crop_rect = box(crop_x1, crop_y1, crop_x2, crop_y2)
    clipped = poly.intersection(crop_rect)

    if clipped.is_empty:
        return []

    if isinstance(clipped, Polygon):
        polygons = [clipped]
    elif isinstance(clipped, MultiPolygon):
        polygons = list(clipped.geoms)
    elif isinstance(clipped, GeometryCollection):
        polygons = [g for g in clipped.geoms if isinstance(g, Polygon)]
    else:
        polygons = []

    new_polygons = []

    for p in polygons:
        if p.is_empty or p.area < MIN_MASK_AREA:
            continue

        exterior = list(p.exterior.coords)[:-1]

        shifted = []
        for x, y in exterior:
            shifted.append((x - crop_x1, y - crop_y1))

        flat = flatten_polygon_coords(shifted)

        if len(flat) >= 6:
            new_polygons.append(flat)

    return new_polygons


def clip_polygon_segmentation_to_crop(segmentation, crop_box):
    clipped_polygons = []

    for poly in segmentation:
        new_polys = clip_single_polygon_to_crop(poly, crop_box)
        clipped_polygons.extend(new_polys)

    return clipped_polygons


def bbox_from_polygons(polygons):
    xs = []
    ys = []

    for poly in polygons:
        for i in range(0, len(poly), 2):
            xs.append(poly[i])
            ys.append(poly[i + 1])

    if not xs or not ys:
        return None

    x_min = min(xs)
    y_min = min(ys)
    x_max = max(xs)
    y_max = max(ys)

    return [
        float(x_min),
        float(y_min),
        float(x_max - x_min),
        float(y_max - y_min)
    ]


def area_from_polygons(polygons):
    return float(sum(polygon_area(poly) for poly in polygons))


# =========================
# 6) RLE / mask 처리 함수
# =========================

def segmentation_to_mask(segmentation, image_height, image_width):
    if isinstance(segmentation, list):
        if len(segmentation) == 0:
            return np.zeros((image_height, image_width), dtype=np.uint8)

        rles = mask_utils.frPyObjects(segmentation, image_height, image_width)
        rle = mask_utils.merge(rles)
        mask = mask_utils.decode(rle)

    elif isinstance(segmentation, dict):
        rle = dict(segmentation)
        counts = rle.get("counts")

        if isinstance(counts, list):
            rle = mask_utils.frPyObjects(rle, image_height, image_width)
        elif isinstance(counts, str):
            rle["counts"] = counts.encode("utf-8")

        mask = mask_utils.decode(rle)

    else:
        raise ValueError(f"Unknown segmentation type: {type(segmentation)}")

    if mask.ndim == 3:
        mask = mask[:, :, 0]

    return mask.astype(np.uint8)


def encode_mask_to_rle(mask):
    mask = mask.astype(np.uint8)
    rle = mask_utils.encode(np.asfortranarray(mask))

    rle["counts"] = rle["counts"].decode("utf-8")
    rle["size"] = [int(mask.shape[0]), int(mask.shape[1])]

    return rle


def mask_to_polygons(mask):
    mask = mask.astype(np.uint8)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    polygons = []

    for contour in contours:
        if len(contour) < 3:
            continue

        contour = contour.squeeze(1)

        if contour.ndim != 2 or contour.shape[0] < 3:
            continue

        area = cv2.contourArea(contour)
        if area < MIN_MASK_AREA:
            continue

        flat = contour.astype(float).flatten().tolist()

        if len(flat) >= 6:
            polygons.append(flat)

    return polygons


def bbox_from_mask(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min = int(xs.min())
    y_min = int(ys.min())
    x_max = int(xs.max())
    y_max = int(ys.max())

    return [
        float(x_min),
        float(y_min),
        float(x_max - x_min + 1),
        float(y_max - y_min + 1)
    ]


# =========================
# 7) annotation crop 통합 함수
# =========================

def crop_annotation_segmentation(segmentation, crop_box, image_height, image_width):
    crop_x1, crop_y1, crop_x2, crop_y2 = crop_box

    if OUTPUT_MASK_FORMAT == "rle":
        full_mask = segmentation_to_mask(segmentation, image_height, image_width)
        cropped_mask = full_mask[crop_y1:crop_y2, crop_x1:crop_x2]

        area = float(cropped_mask.sum())
        if area < MIN_MASK_AREA:
            return None

        new_bbox = bbox_from_mask(cropped_mask)
        if new_bbox is None:
            return None

        new_segmentation = encode_mask_to_rle(cropped_mask)

        return {
            "segmentation": new_segmentation,
            "bbox": new_bbox,
            "area": area
        }

    if isinstance(segmentation, list):
        if OUTPUT_MASK_FORMAT in ["preserve", "polygon"]:
            clipped_polygons = clip_polygon_segmentation_to_crop(segmentation, crop_box)

            if len(clipped_polygons) == 0:
                return None

            new_bbox = bbox_from_polygons(clipped_polygons)
            new_area = area_from_polygons(clipped_polygons)

            if new_bbox is None or new_area < MIN_MASK_AREA:
                return None

            return {
                "segmentation": clipped_polygons,
                "bbox": new_bbox,
                "area": new_area
            }

    if isinstance(segmentation, dict):
        full_mask = segmentation_to_mask(segmentation, image_height, image_width)
        cropped_mask = full_mask[crop_y1:crop_y2, crop_x1:crop_x2]

        area = float(cropped_mask.sum())
        if area < MIN_MASK_AREA:
            return None

        new_bbox = bbox_from_mask(cropped_mask)
        if new_bbox is None:
            return None

        if OUTPUT_MASK_FORMAT == "preserve":
            new_segmentation = encode_mask_to_rle(cropped_mask)

            return {
                "segmentation": new_segmentation,
                "bbox": new_bbox,
                "area": area
            }

        elif OUTPUT_MASK_FORMAT == "polygon":
            polygons = mask_to_polygons(cropped_mask)

            if len(polygons) == 0:
                return None

            new_bbox = bbox_from_polygons(polygons)
            new_area = area_from_polygons(polygons)

            if new_bbox is None or new_area < MIN_MASK_AREA:
                return None

            return {
                "segmentation": polygons,
                "bbox": new_bbox,
                "area": new_area
            }

    return None


# =========================
# 8) split 처리 함수
# =========================

def process_split(split_name):
    input_dir = INPUT_ROOT / split_name
    output_dir = OUTPUT_ROOT / split_name

    ann_path = input_dir / "_annotations.coco.json"

    if not ann_path.exists():
        print(f"[SKIP] {split_name}: _annotations.coco.json 없음")
        return

    connected_dir = output_dir / "connected"
    unconnected_dir = output_dir / "unconnected"

    connected_dir.mkdir(parents=True, exist_ok=True)
    unconnected_dir.mkdir(parents=True, exist_ok=True)

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = coco["images"]
    annotations = coco["annotations"]
    categories = coco["categories"]

    cat_name_to_id = {cat["name"]: cat["id"] for cat in categories}
    cat_id_to_name = {cat["id"]: cat["name"] for cat in categories}

    print(f"\n[{split_name}] class 목록:")
    print(list(cat_name_to_id.keys()))

    if HOOK_CLASS not in cat_name_to_id:
        raise ValueError(
            f"HOOK_CLASS='{HOOK_CLASS}' 클래스가 없음. "
            f"현재 클래스 목록을 보고 HOOK_CLASS를 수정해야 함."
        )

    if LIFELINE_CLASS not in cat_name_to_id:
        raise ValueError(
            f"LIFELINE_CLASS='{LIFELINE_CLASS}' 클래스가 없음. "
            f"현재 클래스 목록을 보고 LIFELINE_CLASS를 수정해야 함."
        )

    hook_class_id = cat_name_to_id[HOOK_CLASS]
    lifeline_class_id = cat_name_to_id[LIFELINE_CLASS]

    image_id_to_annotations = {}
    for ann in annotations:
        image_id_to_annotations.setdefault(ann["image_id"], []).append(ann)

    new_coco = {
        "info": coco.get("info", {}),
        "licenses": coco.get("licenses", []),
        "categories": categories,
        "images": [],
        "annotations": []
    }

    new_image_id = 1
    new_ann_id = 1

    connected_count = 0
    unconnected_count = 0

    for img_info in tqdm(images, desc=f"Processing {split_name}"):
        old_image_id = img_info["id"]
        file_name = img_info["file_name"]

        image_path = resolve_image_path(input_dir, file_name)

        if image_path is None:
            print(f"[WARN] 이미지 파일 못 찾음: {file_name}")
            continue

        anns = image_id_to_annotations.get(old_image_id, [])

        hook_anns = [
            ann for ann in anns
            if ann["category_id"] == hook_class_id
        ]

        if len(hook_anns) == 0:
            continue

        # 핵심 추가 조건:
        # 원본 이미지 안에 lifeline annotation이 하나라도 있으면 connected
        has_lifeline = any(
            ann["category_id"] == lifeline_class_id
            for ann in anns
        )

        status_label = "connected" if has_lifeline else "unconnected"
        save_dir = connected_dir if has_lifeline else unconnected_dir

        with Image.open(image_path) as img:
            img = img.convert("RGB")
            image_width, image_height = img.size

            for hook_idx, hook_ann in enumerate(hook_anns, start=1):
                crop_box = compute_crop_box_from_hook(
                    hook_ann,
                    image_width,
                    image_height
                )

                if crop_box is None:
                    continue

                crop_x1, crop_y1, crop_x2, crop_y2 = crop_box

                cropped_img = img.crop((crop_x1, crop_y1, crop_x2, crop_y2))

                new_width = crop_x2 - crop_x1
                new_height = crop_y2 - crop_y1

                original_stem = Path(file_name).stem
                original_suffix = Path(file_name).suffix

                new_file_name_only = (
                    f"{original_stem}_hook{hook_idx}_ann{hook_ann['id']}_{status_label}_crop"
                    f"{original_suffix}"
                )

                new_image_path = save_dir / new_file_name_only
                cropped_img.save(new_image_path)

                # COCO json에는 split 폴더 기준 상대 경로로 저장
                # 예: connected/xxx.jpg 또는 unconnected/xxx.jpg
                new_file_name_for_json = f"{status_label}/{new_file_name_only}"

                new_img_info = {
                    "id": new_image_id,
                    "file_name": new_file_name_for_json,
                    "width": new_width,
                    "height": new_height,
                    "status": status_label,
                    "source_image": file_name,
                    "source_hook_ann_id": hook_ann["id"]
                }

                new_coco["images"].append(new_img_info)

                if has_lifeline:
                    connected_count += 1
                else:
                    unconnected_count += 1

                # crop 안에 들어오는 annotation 저장
                for ann in anns:
                    if not KEEP_ALL_CLASSES_IN_CROP:
                        if ann["id"] != hook_ann["id"]:
                            continue

                    segmentation = ann.get("segmentation")

                    if segmentation is None:
                        continue

                    try:
                        cropped_ann_data = crop_annotation_segmentation(
                            segmentation=segmentation,
                            crop_box=crop_box,
                            image_height=image_height,
                            image_width=image_width
                        )
                    except Exception as e:
                        print(
                            f"[WARN] annotation 처리 실패: "
                            f"image={file_name}, ann_id={ann.get('id')}, error={e}"
                        )
                        continue

                    if cropped_ann_data is None:
                        continue

                    new_ann = {
                        k: deepcopy(v)
                        for k, v in ann.items()
                        if k not in ["id", "image_id", "segmentation", "bbox", "area"]
                    }

                    new_ann.update({
                        "id": new_ann_id,
                        "image_id": new_image_id,
                        "category_id": ann["category_id"],
                        "segmentation": cropped_ann_data["segmentation"],
                        "bbox": cropped_ann_data["bbox"],
                        "area": cropped_ann_data["area"],
                        "iscrowd": ann.get("iscrowd", 0),
                        "source_ann_id": ann["id"],
                        "crop_status": status_label
                    })

                    new_coco["annotations"].append(new_ann)
                    new_ann_id += 1

                new_image_id += 1

    output_ann_path = output_dir / "_annotations.coco.json"

    with open(output_ann_path, "w", encoding="utf-8") as f:
        json.dump(new_coco, f, ensure_ascii=False, indent=2)

    print(f"\n[DONE] {split_name}")
    print(f"저장 위치: {output_dir}")
    print(f"connected crop 수: {connected_count}")
    print(f"unconnected crop 수: {unconnected_count}")
    print(f"전체 crop 이미지 수: {len(new_coco['images'])}")
    print(f"annotation 수: {len(new_coco['annotations'])}")


# =========================
# 9) 실행
# =========================

for split in ["train", "valid", "test"]:
    process_split(split)

print("\n전체 후크별 connected/unconnected crop 완료")
print(f"최종 저장 위치: {OUTPUT_ROOT}")

In [ ]:
import shutil
from pathlib import Path

folder_path = Path("/content/dyd-safe-2-2-hook-crops-connected")
zip_path = "/content/dyd-safe-2-2-hook-crops-connected"

# zip 생성
shutil.make_archive(zip_path, "zip", folder_path)

print("압축 완료!")
print("저장 위치:", zip_path + ".zip")

In [ ]:
from google.colab import files

files.download("/content/dyd-safe-2-2-hook-crops-connected.zip")

In [ ]:
from google.colab import files

files.download("/content/dyd-safe-2-2-hook-crops-connected")